## Q1. Write a unique paragraph (5–6 sentences) about your favorite topic.

In [ ]:
import re
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

for package in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(package, quiet=True)

text = """I enjoy following the stock market because prices change with news and investor decisions. I read about companies before forming an opinion about their shares. Charts help me notice trends, but they cannot predict every movement. Risk management is important because even a good idea can lose money. I also compare a company’s performance with its industry. Learning about markets has made me more careful with financial decisions."""

clean_text = re.sub(r"[^\w\s]", "", text.lower())
words = word_tokenize(clean_text)
sentences = sent_tokenize(text)

print(clean_text)
print("Words:", words)
print("Sentences:", sentences)
print("split():", text.split())
print("word_tokenize():", word_tokenize(text))

stop_words = set(stopwords.words("english"))
filtered_words = [word for word in words if word not in stop_words]
print("Without stopwords:", filtered_words)

freq = Counter(filtered_words)
print("Word frequency:", freq)
plt.figure(figsize=(8, 4))
plt.bar(freq.keys(), freq.values())
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Q2. Using the same paragraph from Q1

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
filtered_words = [word for word in alphabetic_words if word not in stop_words]

porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [porter.stem(word) for word in filtered_words]
lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

print("Alphabetic words:", alphabetic_words)
print("Without stopwords:", filtered_words)
print("Stemmed:", stemmed_words)
print("Lemmatized:", lemmatized_words)
print("Stemming reduces words to their root-like form, while lemmatization returns dictionary forms. I would use stemming for quick text processing and lemmatization when readable and meaningful words are important.")


## Q3. Choose 3 short texts of your own.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = [
    "Video games combine storytelling, strategy, and quick decision-making.",
    "Competitive gaming rewards practice, teamwork, and clear communication.",
    "Game developers use graphics, sound, and physics to create immersive worlds."
]

count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(texts)
print("Bag of Words")
print(count_vectorizer.get_feature_names_out())
print(count_matrix.toarray())

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF")
print(tfidf_matrix.toarray())

for i, row in enumerate(tfidf_matrix.toarray(), 1):
    top_indices = row.argsort()[::-1][:3]
    print("Text", i, "top keywords:", [(feature_names[j], round(row[j], 3)) for j in top_indices])


## Q4. Write 2 short texts describing two different technologies.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ai_text = """Stock market analysis uses company results and market news.
Investors compare prices, earnings, and business growth.
Charts can show trends but do not guarantee future returns.
Risk management helps limit losses."""

blockchain_text = """Video game design combines art, sound, and programming.
Game engines handle graphics, physics, and player input.
Developers test levels and balance gameplay.
Good design makes a game engaging and easy to understand."""

def preprocess(text):
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())
    return word_tokenize(text)

tokens_ai = preprocess(ai_text)
tokens_blockchain = preprocess(blockchain_text)
set_ai = set(tokens_ai)
set_blockchain = set(tokens_blockchain)

jaccard = len(set_ai & set_blockchain) / len(set_ai | set_blockchain)

vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform([ai_text, blockchain_text])
cosine = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]

print("Stock market tokens:", tokens_ai)
print("Game design tokens:", tokens_blockchain)
print("Jaccard similarity:", jaccard)
print("Cosine similarity:", cosine)
print("Cosine similarity uses TF-IDF weights and considers word importance, while Jaccard treats each unique word equally. In these texts, cosine similarity can be more informative when shared terms have different importance.")


## Q5. Write a short review for a product or service.

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt

reviews = [
    "The laptop is fast, reliable, and easy to use.",
    "The battery is poor and the laptop gets hot.",
    "The product is okay and works as expected.",
    "Excellent performance and a very useful design."
]

positive_reviews = []
for review in reviews:
    blob = TextBlob(review)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    if polarity > 0:
        label = "Positive"
        positive_reviews.append(review)
    elif polarity < 0:
        label = "Negative"
    else:
        label = "Neutral"
    print(review)
    print("Polarity:", polarity, "Subjectivity:", subjectivity, "Class:", label)

cloud_text = " ".join(positive_reviews)
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(cloud_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()


## Q6. Choose your own paragraph (~100 words) as training data.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

training_text = """The stock market allows companies to raise money and investors to buy ownership in businesses. Share prices change because of company results, economic conditions, and investor expectations. Investors often study earnings, revenue, and debt before making decisions. Market charts show past price movements, but they cannot guarantee what will happen next. Diversification can reduce the impact of losses in one investment. Risk management is important because markets can be unpredictable. Learning about companies and financial news helps investors make more informed decisions."""

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
sequences = tokenizer.texts_to_sequences([training_text])[0]
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for i in range(1, len(sequences)):
    input_sequences.append(sequences[:i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = np.eye(total_words)[y]

model = Sequential([
    Embedding(total_words, 32, input_length=max_sequence_len - 1),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X, y, epochs=100, verbose=0)

def generate_text(seed, next_words=12):
    result = seed
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([result])[0]
        sequence = pad_sequences([sequence], maxlen=max_sequence_len - 1, padding="pre")
        prediction = model.predict(sequence, verbose=0)
        next_index = int(np.argmax(prediction))
        next_word = tokenizer.index_word.get(next_index, "")
        if not next_word:
            break
        result += " " + next_word
    return result

print(generate_text("market", 12))
print(generate_text("investors", 12))
print(generate_text("companies", 12))
